# Stage D / NB 21 — figures, tables and exports

Protocol reference: manuscript artifact map (§11); referee point **R1.9** (figures illegible,
tables inconsistent).

## The rule this notebook exists to enforce

**Every result number traces to the identity-matched row of its owning locked artifact.**
That includes NB 17's metrics, NB 19's final comparisons and external results, and the
upstream files that own Tables 1, 6, 10, 11, S1, and S3. The gate fails on any unmatched
numeric cell; a coincidentally equal value in another row is not provenance.

That sounds pedantic until you have shipped a table assembled by hand from four notebooks and
discovered afterwards that one column came from a run that no longer exists.

## One formatting function

`sd.format_number` is the only place a float becomes a string. A metric cannot appear with three
decimals in one table and two in another, and a p-value cannot appear as `0.000` anywhere.

## Best / second-best highlighting is computed, not chosen

Bold for best, underline for second-best, applied by the direction each metric should improve
in (MAE lower, AUROC higher). Choosing the highlight by hand is how the wrong arm ends up bold.

## Outputs (under `stage_D/nb21_exports/`)
`tables/*.csv` and `tables/*.tex`, `figures/*.pdf` and `*.svg`, `captions.md`,
`acronym_table.csv`, `supplementary_results.xlsx`, `figure6_case_panel_manifest.csv`,
`traceability.csv`, `run_config.json`, and `gate_nb21.json`.

## 1. Imports and the locked artifacts

In [ ]:
import hashlib
import json
import math
import random
import re
import shutil
import sys
from collections import OrderedDict, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

# Stage D's own statistics module: bootstrap indices drawn once, DeLong, McNemar, Holm, TOST,
# and the rule that a p-value cannot exist without its metadata.
for _candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent / "stage_D",
                   Path.cwd().parent.parent / "notebooks" / "stage_D"]:
    if (_candidate / "stage_d_stats.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("stage_d_stats.py not found; it must sit beside these notebooks.")
import stage_d_stats as sd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FALLBACK_STAGE_A = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
for _candidate in [FALLBACK_STAGE_A / "nb00_environment" / "stage_a_paths.json",
                   Path.cwd() / "stage_a_paths.json",
                   Path.cwd().parent / "stage_A" / "nb00_environment" / "stage_a_paths.json"]:
    if _candidate.is_file():
        stage_paths = json.loads(_candidate.read_text(encoding="utf-8"))
        print("Path contract:", _candidate)
        break
else:
    raise FileNotFoundError("stage_a_paths.json not found. Run Stage A NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_ROOT = Path(stage_paths["stage_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
STAGE_B_DIR = STAGE_ROOT / "stage_B"
STAGE_C_DIR = STAGE_ROOT / "stage_C"
STAGE_D_DIR = STAGE_ROOT / "stage_D"
STAGE_D_DIR.mkdir(parents=True, exist_ok=True)
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
MODEL_REVISIONS = stage_paths.get("model_revisions", {})

N_FOLDS = 5
N_BOOTSTRAP = sd.BOOTSTRAP_REPLICATES
MAX_SESSION_HOURS = 35.0      # Biowulf limit is 36 h; guard section boundaries
SESSION_DEADLINE = sd.make_session_deadline(MAX_SESSION_HOURS)

print("Stage D output:", STAGE_D_DIR)
print(f"Bootstrap: {N_BOOTSTRAP} patient-level replicates, seed {sd.BOOTSTRAP_SEED}")
print(f"Soft stop: {MAX_SESSION_HOURS:.1f} h after setup; checks occur between sections")

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 21 before code cell 3")

NB21_DIR = STAGE_D_DIR / "nb21_exports"
TABLE_DIR, FIGURE_DIR = NB21_DIR / "tables", NB21_DIR / "figures"
for directory in (NB21_DIR, TABLE_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# NB 21 is a pure export notebook. Remove only its managed outputs at the start so a failed
# rerun cannot leave a passing gate or a figure/table from an older upstream result set.
for pattern in ["*.csv", "*.tex"]:
    for stale in TABLE_DIR.glob(pattern):
        stale.unlink()
for pattern in ["*.pdf", "*.svg", "*.png"]:
    for stale in FIGURE_DIR.glob(pattern):
        stale.unlink()
for name in ["captions.md", "acronym_table.csv", "supplementary_results.xlsx",
             "figure6_case_panel_manifest.csv", "traceability.csv",
             "run_config.json", "gate_nb21.json"]:
    stale = NB21_DIR / name
    if stale.is_file():
        stale.unlink()

NB17_DIR = STAGE_D_DIR / "nb17_statistics"
NB18_DIR = STAGE_D_DIR / "nb18_calibration"
NB19_DIR = STAGE_D_DIR / "nb19_external"
NB20_DIR = STAGE_D_DIR / "nb20_interpretability"
NB16_DIR = STAGE_C_DIR / "nb16_sensitivity"
NB00_DIR = STAGE_A_DIR / "nb00_environment"
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB08_DIR = STAGE_B_DIR / "nb08_biomedclip_entity"
NB09_DIR = STAGE_B_DIR / "nb09_medgemma_lora"
NB10_DIR = STAGE_B_DIR / "nb10_qwen_lora"
NB15_DIR = STAGE_C_DIR / "nb15_reasoner"


def require_upstream_gate(directory, notebook_number):
    candidates = [directory / f"gate_nb{notebook_number:02d}.json",
                  directory / f"gate_nb{notebook_number}.json"]
    path = next((candidate for candidate in candidates if candidate.is_file()), candidates[0])
    if not path.is_file():
        raise FileNotFoundError(
            f"Required upstream gate is missing: {path}. Run NB {notebook_number} to "
            "completion before producing manuscript exports.")
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not bool(payload.get("passed", False)):
        raise RuntimeError(
            f"NB {notebook_number} did not pass its gate: {payload.get('failures', [])}")
    return {"path": str(path), "passed": True, "payload": payload}


UPSTREAM_GATES = {
    "NB17": require_upstream_gate(NB17_DIR, 17),
    "NB18": require_upstream_gate(NB18_DIR, 18),
    "NB19": require_upstream_gate(NB19_DIR, 19),
    "NB20": require_upstream_gate(NB20_DIR, 20),
}


def read_locked(path, owner, required=True):
    path = Path(path)
    if path.is_file():
        return pd.read_csv(path)
    message = (f"{path} not found — {owner} has not produced it. If the directory exists but is "
               "empty the notebook has not finished; if it does not exist at all, check "
               "stage_a_paths.json rather than editing this path.")
    if required:
        raise FileNotFoundError(message)
    print(f"  [absent] {path.name}: {owner} has not run")
    return pd.DataFrame()


ALL_METRICS = read_locked(NB17_DIR / "all_metrics_with_ci.csv", "NB 17")
PAIRED = read_locked(NB19_DIR / "paired_comparisons_final.csv",
                     "NB 19 (final multiplicity pass)")
nb17_config = json.loads((NB17_DIR / "run_config.json").read_text(encoding="utf-8"))
REFERENCE_ARM = nb17_config["reference_arm"]

OPERATING = read_locked(NB18_DIR / "operating_points.csv", "NB 18")
CALIBRATION = read_locked(NB18_DIR / "calibration.csv", "NB 18")
EXTERNAL = read_locked(NB19_DIR / "external_metrics.csv", "NB 19")
E9C = read_locked(NB19_DIR / "e9c_threshold_transfer.csv", "NB 19")
SUBGROUPS = read_locked(NB19_DIR / "subgroup_metrics.csv", "NB 19", required=False)
TAXONOMY = read_locked(NB20_DIR / "failure_taxonomy.csv", "NB 20")
CASE_SELECTION = read_locked(NB20_DIR / "case_selection.csv", "NB 20")
GROUNDING = read_locked(NB20_DIR / "grounding_metrics.csv", "NB 20")
nb19_config = json.loads((NB19_DIR / "run_config.json").read_text(encoding="utf-8"))
nb20_config = json.loads((NB20_DIR / "run_config.json").read_text(encoding="utf-8"))
multiplicity_final = json.loads(
    (NB19_DIR / "multiplicity_families_final.json").read_text(encoding="utf-8"))
kappa_result = json.loads(
    (NB20_DIR / "dual_coding_kappa.json").read_text(encoding="utf-8"))


def require_columns(frame, name, columns):
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise RuntimeError(f"{name} is missing required columns {missing}. Re-run its "
                           "owning notebook before NB 21.")


for frame, name, columns in [
        (ALL_METRICS, "NB 17 all_metrics_with_ci.csv", ["arm", "mrale_mae"]),
        (PAIRED, "NB 19 paired_comparisons_final.csv",
         ["comparison", "arm_a", "arm_b", "endpoint", "family",
          "n_patients", "delta", "ci_low", "ci_high", "test",
          "paired_unit", "reported_p_key", "p_raw", "p_adjusted",
          "p_reportable"]),
        (EXTERNAL, "NB 19 external_metrics.csv", ["cohort", "arm", "n"]),
        (E9C, "NB 19 e9c_threshold_transfer.csv", ["threshold_source"]),
        (TAXONOMY, "NB 20 failure_taxonomy.csv", ["code", "n"]),
        (CASE_SELECTION, "NB 20 case_selection.csv", ["image_key", "filename"])]:
    require_columns(frame, name, columns)

# Cross-notebook identity contract: a passing file beside another run's CSV is not enough.
if str(nb19_config.get("reference_arm")) != str(REFERENCE_ARM):
    raise RuntimeError("NB 19 and NB 17 disagree on the locked reference arm.")
if str(nb20_config.get("reference_arm")) != str(REFERENCE_ARM):
    raise RuntimeError("NB 20 and NB 17 disagree on the locked reference arm.")
expected_bootstrap = str(nb17_config.get("bootstrap_fingerprint") or "").strip()
if not expected_bootstrap:
    raise RuntimeError("NB 17 run_config.json has no bootstrap fingerprint.")
for label, observed in [
        ("NB 19 run configuration", nb19_config.get("nb18_bootstrap_fingerprint")),
        ("NB 19 gate", UPSTREAM_GATES["NB19"]["payload"].get("nb18_bootstrap_fingerprint"))]:
    if str(observed or "").strip() != expected_bootstrap:
        raise RuntimeError(
            f"{label} does not match NB 17's frozen bootstrap fingerprint.")
selection_hash = str(nb20_config.get("selection_hash") or "").strip()
case_fingerprint = str(nb20_config.get("case_table_fingerprint") or "").strip()
if not selection_hash or not case_fingerprint:
    raise RuntimeError(
        "NB 20 predates the fingerprinted qualitative contract. Re-run the updated NB 20.")
for column, expected_value in [("selection_hash", selection_hash),
                               ("case_table_fingerprint", case_fingerprint)]:
    if column not in CASE_SELECTION:
        raise RuntimeError(f"NB 20 case_selection.csv lacks {column}.")
    observed_values = set(CASE_SELECTION[column].dropna().astype(str))
    if observed_values != {expected_value}:
        raise RuntimeError(
            f"NB 20 {column} mismatch: {sorted(observed_values)} vs {expected_value}.")
    gate_value = str(UPSTREAM_GATES["NB20"]["payload"].get(column) or "").strip()
    if gate_value != expected_value:
        raise RuntimeError(f"NB 20 gate disagrees on {column}.")
SENSITIVITY = read_locked(NB16_DIR / "e6_sensitivity_grid.csv", "NB 16 (Stage C)",
                          required=False)

# Operational measurements use several historical schemas. Arm summaries own the rows;
# run_config.json owns fold-level training time / peak memory, and NB 16 owns reasoner
# latency / token counts. Normalize those names here without inventing a measurement.
operational_frames = []
for root in [STAGE_B_DIR, STAGE_C_DIR]:
    for path in sorted(root.rglob("arm_summary.csv")):
        try:
            frame = pd.read_csv(path)
        except (OSError, pd.errors.ParserError):
            continue
        if len(frame) and "arm" in frame.columns:
            frame = frame.copy()
            frame["operational_source"] = str(path)
            config_path = path.parent / "run_config.json"
            if config_path.is_file():
                payload = json.loads(config_path.read_text(encoding="utf-8"))
                by_arm = defaultdict(list)
                diagnostics = payload.get("fold_diagnostics") or {}
                for key, item in diagnostics.items():
                    if not isinstance(item, dict):
                        continue
                    prefix = str(key).split("/fold")[0]
                    candidates = [prefix, f"E0g_{prefix}"]
                    arm = next((candidate for candidate in candidates
                                if candidate in set(frame["arm"].astype(str))), None)
                    if arm is not None:
                        by_arm[arm].append(item)
                fold_summaries = payload.get("fold_summaries") or {}
                if len(frame) == 1 and isinstance(fold_summaries, dict):
                    only_arm = str(frame.iloc[0]["arm"])
                    by_arm[only_arm].extend(
                        item for item in fold_summaries.values() if isinstance(item, dict))

                def finite_values(items, aliases):
                    values = []
                    for item in items:
                        for alias in aliases:
                            value = pd.to_numeric(pd.Series([item.get(alias)]),
                                                  errors="coerce").iloc[0]
                            if pd.notna(value) and math.isfinite(float(value)):
                                values.append(float(value)); break
                    return values

                for arm, items in by_arm.items():
                    row_mask = frame["arm"].astype(str) == arm
                    peak = finite_values(items, ["peak_memory_gib", "peak_memory_gb",
                                                  "peak_gpu_memory_gb"])
                    seconds = finite_values(items, ["train_seconds", "training_seconds"])
                    total = finite_values(items, ["total_parameters", "parameter_count"])
                    trainable = finite_values(items, ["trainable_parameters"])
                    if peak:
                        frame.loc[row_mask, "peak_memory_gib"] = max(peak)
                    if seconds:
                        frame.loc[row_mask, "median_train_seconds_per_fold"] = float(
                            np.median(seconds))
                    if total and len(set(total)) == 1:
                        frame.loc[row_mask, "total_parameters"] = total[0]
                    if trainable and len(set(trainable)) == 1:
                        frame.loc[row_mask, "trainable_parameters"] = trainable[0]
                    if peak or seconds or total or trainable:
                        frame.loc[row_mask, "operational_resource_source"] = str(config_path)
            operational_frames.append(frame)
OPERATIONAL_RAW = (pd.concat(operational_frames, ignore_index=True, sort=False)
                   if operational_frames else pd.DataFrame())

# NB 15 stores per-image latency and token counts in its fingerprinted append-only journals.
# Aggregate those completed outer-test rows here; this is a lightweight export repair and
# never reloads a model. Duplicate arm/image rows are rejected rather than averaged twice.
reasoner_operational_rows = {}
reasoner_operational_sources = defaultdict(set)
journal_dir = NB15_DIR / "journals"
if journal_dir.is_dir():
    for journal_path in sorted(journal_dir.glob("*.jsonl")):
        stamp_path = journal_path.with_suffix(".fingerprint.json")
        if not stamp_path.is_file():
            continue
        stamp = json.loads(stamp_path.read_text(encoding="utf-8"))
        expected_fingerprint = str(stamp.get("fingerprint") or "")
        with journal_path.open(encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, start=1):
                if not line.strip():
                    continue
                try:
                    row = json.loads(line)
                except json.JSONDecodeError as exc:
                    raise RuntimeError(
                        f"Malformed NB 15 journal row {journal_path}:{line_number}") from exc
                extra = row.get("extra") or {}
                if str(extra.get("fingerprint") or "") != expected_fingerprint:
                    continue
                if str(extra.get("score_role") or "outer_test") != "outer_test":
                    continue
                arm, image_key = str(row.get("arm") or ""), str(row.get("image_key") or "")
                if not arm or not image_key:
                    continue
                key = (arm, image_key)
                entry = {"seconds": row.get("seconds"),
                         "tokens": extra.get("n_completion_tokens")}
                if key in reasoner_operational_rows and reasoner_operational_rows[key] != entry:
                    raise RuntimeError(
                        f"Conflicting active NB 15 journal rows for {arm}/{image_key}.")
                reasoner_operational_rows[key] = entry
                reasoner_operational_sources[arm].add(str(journal_path))

reasoner_summary_rows = []
for arm in sorted({key[0] for key in reasoner_operational_rows}):
    rows = [value for (row_arm, _), value in reasoner_operational_rows.items()
            if row_arm == arm]
    seconds = pd.to_numeric(pd.Series([row["seconds"] for row in rows]),
                            errors="coerce").dropna()
    tokens = pd.to_numeric(pd.Series([row["tokens"] for row in rows]),
                           errors="coerce").dropna()
    reasoner_summary_rows.append({
        "arm": arm, "median_seconds": (float(seconds.median()) if len(seconds) else None),
        "p95_seconds": (float(seconds.quantile(0.95)) if len(seconds) else None),
        "mean_completion_tokens": (float(tokens.mean()) if len(tokens) else None),
        "n_operational_images": len(rows),
        "operational_latency_source": ";".join(sorted(reasoner_operational_sources[arm]))})
REASONER_OPERATIONAL = pd.DataFrame(reasoner_summary_rows)

COHORT_TABLE = read_locked(NB03_DIR / "cohort_composition_table1_full.csv",
                           "NB 03 (Table 1)", required=False)
COHORT_TABLE_SOURCE = "cohort_composition_table1_full.csv (NB 03)"
if not len(COHORT_TABLE):
    COHORT_TABLE = read_locked(NB02_DIR / "cohort_composition_table1.csv",
                               "NB 02 (internal Table 1)", required=False)
    COHORT_TABLE_SOURCE = "cohort_composition_table1.csv (NB 02; internal only)"
PROBE_COEFFICIENTS = read_locked(NB08_DIR / "probe_coefficients.csv",
                                 "NB 08 (Table 11)", required=False)
MODEL_REGISTRY = read_locked(NB00_DIR / "model_registry.csv",
                             "NB 00 (Table S3)", required=False)
hyperparameter_rows = []
for notebook, path in [("NB09", NB09_DIR / "sweep_selection.json"),
                       ("NB10", NB10_DIR / "sweep_selection.json")]:
    if not path.is_file():
        continue
    payload = json.loads(path.read_text(encoding="utf-8"))
    chosen = payload.get("final_config") or payload.get("chosen") or {}
    families = payload.get("families") or {}
    swept_keys = {key for details in families.values() for key in (details.get("best") or {})}
    for parameter, value in sorted(chosen.items()):
        supporting = [family for family, details in families.items()
                      if parameter in (details.get("best") or {})]
        hyperparameter_rows.append({
            "Notebook": notebook, "Parameter": parameter, "Value": value,
            "Provenance": ("sweep-selected" if parameter in swept_keys else
                           "inherited tested default"),
            "Evidence": ";".join(supporting) or "base_config",
            "Source": str(path)})
HYPERPARAMETERS = pd.DataFrame(hyperparameter_rows)

print(f"\nLocked artifacts loaded. Reference arm: {REFERENCE_ARM}")
print(f"  all_metrics_with_ci.csv : {len(ALL_METRICS)} arms")
print(f"  paired_comparisons_final.csv: {len(PAIRED)} comparisons")

# Every value that may legally appear in a table, for the traceability gate.
#
# A printed number is a ROUNDED form of a locked value: format_number gives 3 decimals, so
# 1.1099 prints as "1.110". Storing only the full-precision value and comparing strings would
# fail every cell for a reason that has nothing to do with provenance. The index therefore
# holds each source value rounded to every precision a table can display it at.
DISPLAY_PRECISIONS = [0, 1, 2, 3, 4]
TRACEABLE = {places: {} for places in DISPLAY_PRECISIONS}


def index_source(frame, source_name):
    """Add every finite numeric value in `frame` to the traceability index."""
    if not len(frame):
        return 0
    n = 0
    for column in frame.columns:
        for value in pd.to_numeric(frame[column], errors="coerce").dropna():
            value = float(value)
            if not math.isfinite(value):
                continue
            n += 1
            for places in DISPLAY_PRECISIONS:
                # Sign-flipped deltas are the same measurement read the other way round.
                for candidate in (round(value, places), round(-value, places)):
                    TRACEABLE[places].setdefault(candidate, source_name)
    return n


# Every LOCKED upstream artifact is a legitimate source. The rule is not "everything comes from
# NB 17" -- Tables 8 and 9 legitimately come from NB 19 -- it is "every number came from a
# notebook that computed and saved it", so nothing can be typed here. The index records WHICH
# artifact each value came from, which is stronger provenance than a single-file rule.
LOCKED_SOURCES = OrderedDict([
    ("all_metrics_with_ci.csv (NB 17)", ALL_METRICS),
    ("paired_comparisons_final.csv (NB 19)", PAIRED),
    ("operating_points.csv (NB 18)", OPERATING),
    ("calibration.csv (NB 18)", CALIBRATION),
    ("external_metrics.csv (NB 19)", EXTERNAL),
    ("e9c_threshold_transfer.csv (NB 19)", E9C),
    ("subgroup_metrics.csv (NB 19)", SUBGROUPS),
    ("failure_taxonomy.csv (NB 20)", TAXONOMY),
    ("case_selection.csv (NB 20)", CASE_SELECTION),
    ("grounding_metrics.csv (NB 20)", GROUNDING),
    ("e6_sensitivity_grid.csv (NB 16)", SENSITIVITY),
    ("fingerprinted outer-test journals (NB 15)", REASONER_OPERATIONAL),
    ("operational arm summaries + run configs (NB 05-15)", OPERATIONAL_RAW),
    (COHORT_TABLE_SOURCE, COHORT_TABLE),
    ("probe_coefficients.csv (NB 08)", PROBE_COEFFICIENTS),
    ("model_registry.csv (NB 00)", MODEL_REGISTRY),
    ("sweep_selection.json files (NB 09-10)", HYPERPARAMETERS),
])
print("\nTraceability index:")
for name, frame in LOCKED_SOURCES.items():
    print(f"  {index_source(frame, name):>6,} values from {name}")
print(f"  {len(TRACEABLE[3]):,} distinct values at 3 decimals")

## 2. Formatting and highlighting

`sd.format_number` is the single conversion point. `highlight` marks best and second-best by the
direction each metric improves in, so the bold cell is chosen by the data rather than by whoever
assembled the table last.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 21 before code cell 5")

# Direction each metric should improve in. Anything absent is not highlighted, which is safer
# than guessing.
LOWER_IS_BETTER = {"mrale_mae", "mrale_rmse", "brier", "ece", "median_seconds", "p95_seconds",
                   "severity_mae", "false_positive_rate", "mae"}
HIGHER_IS_BETTER = {"covid_auroc", "covid_auprc", "mrale_qwk", "mrale_spearman_rho",
                    "covid_balanced_accuracy", "covid_sensitivity", "covid_specificity",
                    "mrale_within1_accuracy", "mrale_coverage", "specificity", "auroc",
                    "spearman_rho", "qwk_after_rank_mapping", "sensitivity"}

KIND_OF = {"covid_auroc": "auroc", "auroc": "auroc", "mrale_qwk": "kappa",
           "mrale_coverage": "percent", "median_seconds": "seconds", "p95_seconds": "seconds",
           "n_images": "count", "n_patients": "count", "n": "count"}


def cell(frame, column, kind=None):
    """One column formatted, with its interval attached where NB 17 recorded one."""
    kind = kind or KIND_OF.get(column, "metric")
    low, high = f"{column}_ci_low", f"{column}_ci_high"
    if low in frame.columns and high in frame.columns:
        return [sd.format_interval(row[column], row[low], row[high], kind)
                for _, row in frame.iterrows()]
    return [sd.format_number(v, kind) for v in frame[column]]


def highlight(frame, formatted, column):
    """Bold the best value and underline the second best, by the metric's own direction."""
    if column not in LOWER_IS_BETTER and column not in HIGHER_IS_BETTER:
        return formatted
    values = pd.to_numeric(frame[column], errors="coerce")
    if values.notna().sum() < 2:
        return formatted
    finite = values.dropna()
    distinct = sorted(finite.unique(), reverse=column in HIGHER_IS_BETTER)
    best_value = distinct[0]
    second_value = distinct[1] if len(distinct) > 1 else None
    out = list(formatted)
    for position, index in enumerate(frame.index):
        if pd.notna(values.loc[index]) and math.isclose(
                float(values.loc[index]), float(best_value), rel_tol=0.0, abs_tol=1e-12):
            out[position] = f"**{out[position]}**"
        elif second_value is not None and pd.notna(values.loc[index]) and math.isclose(
                float(values.loc[index]), float(second_value), rel_tol=0.0, abs_tol=1e-12):
            out[position] = f"_{out[position]}_"
    return out


def to_latex(frame, caption, label):
    """Minimal booktabs LaTeX. Markdown emphasis is translated, not stripped."""
    def escape(text):
        text = str(text)
        for source, target in [("\\", r"\textbackslash{}"), ("_", r"\_"), ("&", r"\&"),
                               ("%", r"\%"), ("#", r"\#")]:
            text = text.replace(source, target)
        return text

    def render(value):
        text = str(value)
        if text.startswith("**") and text.endswith("**"):
            return r"\textbf{" + escape(text[2:-2]) + "}"
        if text.startswith("_") and text.endswith("_") and len(text) > 2:
            return r"\underline{" + escape(text[1:-1]) + "}"
        return escape(text)

    lines = [r"\begin{table}[t]", r"\centering", r"\caption{" + escape(caption) + "}",
             r"\label{" + label + "}",
             r"\begin{tabular}{l" + "r" * (len(frame.columns) - 1) + "}", r"\toprule",
             " & ".join(escape(c) for c in frame.columns) + r" \\", r"\midrule"]
    for _, row in frame.iterrows():
        lines.append(" & ".join(render(v) for v in row) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)


generated_tables, captions = OrderedDict(), OrderedDict()


def emit(name, frame, caption, label):
    frame.to_csv(TABLE_DIR / f"{name}.csv", index=False)
    (TABLE_DIR / f"{name}.tex").write_text(to_latex(frame, caption, label), encoding="utf-8")
    generated_tables[name] = frame
    captions[name] = caption
    print(f"  {name}: {len(frame)} rows x {len(frame.columns)} columns")


print("Formatting helpers ready. Bold = best, underline = second best, by metric direction.")

## 3. Tables 2–5 and 7 — the main results

Built from `all_metrics_with_ci.csv` and NB 19's final multiplicity-adjusted comparisons.

**Coverage sits beside every mRALE column** (§7.4), and confusion counts sit beside every
detection rate. Those two habits are what the previous submission lacked, and they are what a
reviewer checks first.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 21 before code cell 7")

# ---- Table 2: the main comparison ---------------------------------------------------------
main = ALL_METRICS.copy()
if "family" in main.columns:
    main = main[main["family"].isin(["E0", "E1", "E7"])]
main = main.sort_values("mrale_mae").reset_index(drop=True)
table2 = pd.DataFrame({"Arm": main["arm"]})
table2["mRALE MAE [95% CI]"] = highlight(main, cell(main, "mrale_mae"), "mrale_mae")
table2["Coverage"] = [sd.format_number(v, "percent") for v in main.get(
    "mrale_coverage", pd.Series([np.nan] * len(main)))]
if "mrale_qwk" in main.columns:
    table2["QWK"] = highlight(main, cell(main, "mrale_qwk"), "mrale_qwk")
if "covid_auroc" in main.columns:
    table2["COVID AUROC [95% CI]"] = highlight(main, cell(main, "covid_auroc"), "covid_auroc")
if "covid_balanced_accuracy" in main.columns:
    table2["Balanced acc."] = [sd.format_number(v) for v in main["covid_balanced_accuracy"]]
for column, header in [("covid_tp", "TP"), ("covid_fp", "FP"), ("covid_tn", "TN"),
                       ("covid_fn", "FN")]:
    if column in main.columns:
        table2[header] = [sd.format_number(v, "count") for v in main[column]]
emit("table2_main_comparison", table2,
     "Main comparison on the pooled out-of-fold internal cohort. mRALE MAE applies the "
     "invalid-output penalty of protocol 7.4 to every arm; coverage is the fraction of images "
     "with a parseable prediction and is reported beside every valid-only figure. Confusion "
     "counts accompany every detection rate. Intervals are patient-level bootstrap percentile "
     f"intervals ({N_BOOTSTRAP} replicates, seed {sd.BOOTSTRAP_SEED}). Best in bold, "
     "second best underlined.", "tab:main")

# ---- Table 3: E1 roster ablation -----------------------------------------------------------
roster = ALL_METRICS[ALL_METRICS["arm"].astype(str).str.startswith("E1")].copy()
if len(roster):
    roster = roster.sort_values("arm").reset_index(drop=True)
    table3 = pd.DataFrame({"Roster arm": roster["arm"]})
    table3["mRALE MAE [95% CI]"] = highlight(roster, cell(roster, "mrale_mae"), "mrale_mae")
    if "covid_auroc" in roster.columns:
        table3["COVID AUROC"] = highlight(roster, cell(roster, "covid_auroc"), "covid_auroc")
    deltas, tests = [], []
    for arm in roster["arm"]:
        row = PAIRED[(PAIRED.get("arm_a") == arm)
                     & (PAIRED.get("arm_b") == REFERENCE_ARM)
                     & (PAIRED.get("family") == "F2")
                     & (PAIRED.get("endpoint") == "mRALE MAE")
                     & (PAIRED.get("reported_p_key") == "p_bootstrap")] if len(PAIRED) \
            else pd.DataFrame()
        if len(row) > 1:
            raise RuntimeError(
                f"Table 3 has {len(row)} final paired-bootstrap rows for {arm!r}.")
        if len(row):
            entry = row.iloc[0]
            deltas.append(sd.format_interval(entry["delta"], entry["ci_low"],
                                             entry["ci_high"], "delta"))
            p = entry.get("p_adjusted", entry.get("p_raw"))
            tests.append(sd.format_number(p, "p") if pd.notna(p) else "--")
        else:
            deltas.append("--"); tests.append("--")
    table3[f"Paired delta vs {REFERENCE_ARM}"] = deltas
    table3["Holm-adjusted p"] = tests
    emit("table3_roster_ablation", table3,
         "E1 agent-roster ablation. Deltas are patient-level paired differences in penalised "
         f"mRALE MAE against {REFERENCE_ARM}, computed on the images both arms scored; "
         "positive means the arm is worse. p-values are Holm-adjusted within family F2. An "
         "interval straddling zero is inconclusive, not evidence of equivalence — see the "
         "equivalence tests in the supplement.", "tab:roster")

# ---- Table 4: E4 localization --------------------------------------------------------------
localization = ALL_METRICS[ALL_METRICS.get("family") == "E4"].copy() \
    if "family" in ALL_METRICS.columns else pd.DataFrame()
if len(localization):
    localization = localization.sort_values("mrale_mae").reset_index(drop=True)
    table4 = pd.DataFrame({"View / arm": localization["arm"]})
    table4["mRALE MAE [95% CI]"] = highlight(localization, cell(localization, "mrale_mae"),
                                             "mrale_mae")
    table4["Coverage"] = [sd.format_number(v, "percent")
                          for v in localization.get("mrale_coverage",
                                                    pd.Series([np.nan] * len(localization)))]
    if "mrale_qwk" in localization.columns:
        table4["QWK"] = cell(localization, "mrale_qwk")
    emit("table4_localization", table4,
         "E4 anatomy-aware localization ablation. V0 is the whole radiograph, V1 the thorax "
         "crop, V2 the per-lung masked views. Oracle and heuristic bounds, where computed, "
         "bracket what localization quality can contribute.", "tab:localization")

# ---- Table 5: fusion vs reasoning -----------------------------------------------------------
fusion = ALL_METRICS[ALL_METRICS["arm"].astype(str).str.startswith("E7")].copy()
if len(fusion):
    fusion = pd.concat([fusion, ALL_METRICS[ALL_METRICS["arm"] == REFERENCE_ARM]],
                       ignore_index=True).drop_duplicates("arm")
    fusion = fusion.sort_values("mrale_mae").reset_index(drop=True)
    table5 = pd.DataFrame({"Aggregation method": fusion["arm"]})
    table5["mRALE MAE [95% CI]"] = highlight(fusion, cell(fusion, "mrale_mae"), "mrale_mae")
    if "covid_auroc" in fusion.columns:
        table5["COVID AUROC"] = highlight(fusion, cell(fusion, "covid_auroc"), "covid_auroc")
    table5["Images"] = [sd.format_number(v, "count") for v in fusion["n_images"]]
    emit("table5_fusion_vs_reasoning", table5,
         "E7 fusion baselines against LLM reasoning. All arms consume the same structured "
         "agent outputs. The decisive comparison is the reasoning arm against learned "
         "stacking (E7d) on identical inputs; see Table 7 for the paired test. Note the image "
         "count: fusion arms are fitted on the intersection of images every agent scored, and "
         "are not directly comparable to a single agent evaluated on the full cohort.",
         "tab:fusion")

# ---- Table 7: paired tests -------------------------------------------------------------------
if len(PAIRED):
    tests = PAIRED.copy()
    table7 = pd.DataFrame({
        "Comparison": tests["comparison"], "Endpoint": tests["endpoint"],
        "Family": tests["family"],
        "n (patients)": [sd.format_number(v, "count") for v in tests["n_patients"]],
        "Effect [95% CI]": [sd.format_interval(r["delta"], r["ci_low"], r["ci_high"], "delta")
                            for _, r in tests.iterrows()],
        "Test": tests.get("test", pd.Series(["--"] * len(tests))),
        "Paired unit": tests.get("paired_unit", pd.Series(["--"] * len(tests))),
        "p (raw)": [sd.format_number(v, "p") for v in tests["p_raw"]],
        "p (Holm)": [sd.format_number(v, "p") if pd.notna(v) else "unadjusted"
                     for v in tests.get("p_adjusted", pd.Series([np.nan] * len(tests)))],
    })
    emit("table7_paired_tests", table7,
         "Paired hypothesis tests. Every row states its test, its paired unit (the patient, "
         "protocol 8.1), its multiplicity family and whether the p-value is adjusted "
         "(protocol 8.8). Holm-Bonferroni is applied within each pre-declared family; "
         "exploratory comparisons are reported unadjusted and labelled as such.", "tab:tests")

## 4. Tables 6, 8, 9, 10 and S2

Sensitivity (exploratory, labelled), external validation, subgroups, operational cost, and the
augmentation policy including the forbidden list.

**Table 10 belongs in the paper.** A framework that needs six model loads per image should say
so — protocol §7.3 asks for parameter counts, memory, latency and token counts precisely so the
cost of the approach is visible next to its accuracy.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 21 before code cell 9")

# ---- Table 6: sensitivity (exploratory) ----------------------------------------------------
if len(SENSITIVITY):
    sensitivity = SENSITIVITY.copy()
    columns = [c for c in ["arm", "family", "mae", "qwk", "valid_rate", "median_seconds",
                           "mean_completion_tokens"] if c in sensitivity.columns]
    sensitivity = sensitivity[columns].sort_values("mae").reset_index(drop=True)
    table6 = pd.DataFrame({"Arm": sensitivity["arm"],
                           "Family": sensitivity.get("family", "E6")})
    table6["mRALE MAE"] = [sd.format_number(v) for v in sensitivity["mae"]]
    for source, header, kind in [("qwk", "QWK", "kappa"), ("valid_rate", "Valid JSON", "metric"),
                                 ("median_seconds", "Median s/case", "seconds"),
                                 ("mean_completion_tokens", "Tokens", "count")]:
        if source in sensitivity.columns:
            table6[header] = [sd.format_number(v, kind) for v in sensitivity[source]]
    emit("table6_sensitivity_exploratory", table6,
         "E5/E6 sensitivity summary. DECLARED EXPLORATORY (protocol 8.6): reported with effect "
         "sizes and unadjusted p-values, never as confirmatory evidence. Accuracy appears "
         "beside latency and token count because an arm that buys 0.1 MAE for five times the "
         "compute is not an improvement.", "tab:sensitivity")

# ---- Table 8: external validation -----------------------------------------------------------
if len(EXTERNAL):
    external = EXTERNAL.copy()
    table8 = pd.DataFrame({"Cohort": external["cohort"],
                           "Sub-cohort": external.get("subcohort", ""),
                           "Arm": external["arm"], "Adapter": external.get("adapter", "")})
    table8["n"] = [sd.format_number(v, "count") for v in external["n"]]
    for source, header, kind in [("specificity", "Specificity", "metric"),
                                 ("false_positive_rate", "FPR", "metric"),
                                 ("auroc", "AUROC", "auroc"),
                                 ("sensitivity", "Sensitivity", "metric"),
                                 ("spearman_rho", "Spearman rho", "metric"),
                                 ("qwk_after_rank_mapping", "QWK (rank-mapped)", "kappa")]:
        if source in external.columns:
            table8[header] = [sd.format_number(v, kind) for v in external[source]]
    for source, low, high, header, kind in [
            ("specificity", "specificity_ci_low", "specificity_ci_high",
             "Specificity [95% CI]", "metric"),
            ("auroc", "auroc_ci_low", "auroc_ci_high",
             "AUROC [95% CI]", "auroc"),
            ("severity_mae", "severity_mae_ci_low", "severity_mae_ci_high",
             "Severity MAE [95% CI]", "metric")]:
        if all(column in external.columns for column in [source, low, high]):
            table8[header] = [sd.format_interval(r[source], r[low], r[high], kind)
                              for _, r in external.iterrows()]
    emit("table8_external_validation", table8,
         "External validation. X1 contains no PCR positives, so AUROC and sensitivity are "
         "undefined there and are left blank rather than imputed. X3 uses a different severity "
         "rubric, so rank agreement is reported and MAE deliberately is not (protocol E9d). "
         "Every cohort passed the membership guard before being scored. Threshold-dependent "
         "X2 headline results at the internally selected operating point appear in Table 8b; "
         "this table otherwise reports the decision stored by each upstream external run.",
         "tab:external")

# ---- Table 8b: X2 operating point transferred from internal inner validation ------------
e9c_headline = E9C[
    E9C["threshold_source"].astype(str).str.contains("HEADLINE", na=False)].copy()
if len(e9c_headline):
    if e9c_headline.duplicated(["cohort", "arm"]).any():
        raise RuntimeError("NB 19 contains duplicate E9c headline rows for an arm/cohort.")
    e9c_headline = e9c_headline.sort_values(["cohort", "arm"]).reset_index(drop=True)
    table8b = pd.DataFrame({
        "Cohort": e9c_headline["cohort"], "Arm": e9c_headline["arm"],
        "Adapter provenance": e9c_headline["adapter_source"],
        "n": [sd.format_number(v, "count") for v in e9c_headline["n"]],
        "Transferred threshold": [sd.format_number(v) for v in e9c_headline["threshold"]],
        "External AUROC": [sd.format_number(v, "auroc") for v in e9c_headline["auroc"]],
        "Internal AUROC": [sd.format_number(v, "auroc") for v in e9c_headline["internal_auroc"]],
        "AUROC decrease": [sd.format_number(v, "delta")
                              for v in e9c_headline["auroc_drop_vs_internal"]],
        "Sensitivity": [sd.format_number(v) for v in e9c_headline["sensitivity"]],
        "Specificity": [sd.format_number(v) for v in e9c_headline["specificity"]],
        "Balanced acc.": [sd.format_number(v) for v in e9c_headline["balanced_accuracy"]],
        "TP": [sd.format_number(v, "count") for v in e9c_headline["tp"]],
        "FP": [sd.format_number(v, "count") for v in e9c_headline["fp"]],
        "TN": [sd.format_number(v, "count") for v in e9c_headline["tn"]],
        "FN": [sd.format_number(v, "count") for v in e9c_headline["fn"]],
    })
    emit("table8b_x2_threshold_transfer", table8b,
         "X2 cross-site operating-point transfer. Thresholds were selected exclusively "
         "from internal fold-specific inner-validation scores in NB 18 and transferred "
         "unchanged by NB 19. The locally re-tuned X2 upper bound is deliberately excluded. "
         "Adapter provenance distinguishes a predeclared fold-0 model from a fixed upstream "
         "ensemble.", "tab:external-threshold")

# ---- Table 9: subgroups -----------------------------------------------------------------------
if len(SUBGROUPS):
    subgroups = SUBGROUPS.copy()
    table9 = pd.DataFrame({"Field": subgroups["field"], "Value": subgroups["value"]})
    table9["Images"] = [sd.format_number(v, "count") for v in subgroups["n_images"]]
    table9["Patients"] = [sd.format_number(v, "count") for v in subgroups["n_patients"]]
    table9["mRALE MAE [95% CI]"] = [
        sd.format_interval(r["mrale_mae"], r.get("mae_ci_low"), r.get("mae_ci_high"))
        for _, r in subgroups.iterrows()]
    if "balanced_accuracy" in subgroups.columns:
        table9["Balanced acc."] = [sd.format_number(v) for v in subgroups["balanced_accuracy"]]
    if "excludes_pooled_mae" in subgroups.columns:
        table9["Excludes pooled"] = ["yes" if v is True or str(v).lower() == "true" else ""
                                     for v in subgroups["excludes_pooled_mae"]]
    emit("table9_subgroups", table9,
         "Subgroup analysis within the internal cohort. Intervals are patient-level bootstrap. "
         "A subgroup marked as excluding the pooled estimate is a descriptive observation, not "
         "a multiplicity-controlled finding: across this many subgroups some separation is "
         "expected by chance. Age band and portable-vs-fixed acquisition are absent from the "
         "cohort metadata and could not be computed.", "tab:subgroups")

# ---- Table 10: operational cost ---------------------------------------------------------------
# Merge accuracy-owned latency with the NB 09--15 arm summaries. Missing required fields stay
# visibly blank and are a release-gate failure; they are never silently relegated to configs.
operational = ALL_METRICS.copy()
operational_ambiguities = {}
source = pd.DataFrame(columns=["arm"])
if len(OPERATIONAL_RAW):
    operational_ambiguities = {
        str(arm): {"n_rows": int(len(group)),
                   "sources": sorted(set(group["operational_source"].astype(str)))}
        for arm, group in OPERATIONAL_RAW.groupby("arm") if len(group) > 1}
    source = OPERATIONAL_RAW.sort_values("operational_source").drop_duplicates("arm", keep="last")
supplement_frames = []
if len(REASONER_OPERATIONAL):
    supplement_frames.append(REASONER_OPERATIONAL.copy())
if len(SENSITIVITY):
    sensitivity_supplement = SENSITIVITY.copy()
    if len(REASONER_OPERATIONAL):
        sensitivity_supplement = sensitivity_supplement[
            ~sensitivity_supplement["arm"].isin(set(REASONER_OPERATIONAL["arm"]))]
    sensitivity_supplement["operational_latency_source"] = str(
        NB16_DIR / "e6_sensitivity_grid.csv")
    supplement_frames.append(sensitivity_supplement)
if supplement_frames:
    supplement = pd.concat(supplement_frames, ignore_index=True, sort=False)
    supplement_columns = [column for column in
                          ["arm", "median_seconds", "p95_seconds",
                           "mean_completion_tokens", "n_operational_images",
                           "operational_latency_source"]
                          if column in supplement.columns]
    supplement = supplement[supplement_columns].copy()
    if supplement["arm"].astype(str).duplicated().any():
        duplicates = supplement.loc[supplement["arm"].astype(str).duplicated(False),
                                    "arm"].astype(str).unique()[:5].tolist()
        raise RuntimeError(f"NB 16 operational supplement has duplicate arms: {duplicates}")
    source = source.merge(supplement, on="arm", how="outer",
                          suffixes=("", "_nb16"), validate="one_to_one")
    for column in supplement_columns:
        if column == "arm":
            continue
        extra = f"{column}_nb16"
        if extra in source:
            source[column] = source[column].combine_first(source[extra])
            source = source.drop(columns=extra)
if len(source):
    operational = operational.merge(source, on="arm", how="left", suffixes=("", "_summary"))
# Preserve the actual locked row before adding display-normalized helper columns below.
# Traceability must never validate a derived cell against the derived cell itself.
operational_locked = operational.copy()

def first_column(frame, aliases):
    for alias in aliases:
        if alias in frame.columns:
            return frame[alias]
    return pd.Series([np.nan] * len(frame), index=frame.index)

if len(operational):
    operational = operational.copy()
    operational["op_total_parameters"] = first_column(operational,
        ["total_parameters", "parameter_count", "parameters_total"])
    if "total_parameters_M" in operational.columns:
        mask = pd.to_numeric(operational["op_total_parameters"], errors="coerce").isna()
        operational.loc[mask, "op_total_parameters"] = (
            pd.to_numeric(operational.loc[mask, "total_parameters_M"],
                          errors="coerce") * 1e6)
    operational["op_trainable_parameters"] = first_column(operational,
        ["trainable_parameters", "trainable_parameters_M", "trainable_M"])
    if "trainable_parameters_M" in operational.columns:
        mask = operational["trainable_parameters"].isna() if "trainable_parameters" in operational else pd.Series(True, index=operational.index)
        operational.loc[mask, "op_trainable_parameters"] = (
            pd.to_numeric(operational.loc[mask, "trainable_parameters_M"],
                          errors="coerce") * 1e6)
    operational["op_peak_gpu_gb"] = first_column(operational,
        ["peak_gpu_memory_gb", "peak_memory_gb", "peak_memory_gib",
         "max_memory_gb"])
    operational["op_training_hours"] = first_column(operational,
        ["training_hours", "wall_clock_training_hours", "train_hours"])
    train_seconds = first_column(operational,
        ["median_train_seconds_per_fold", "train_seconds_per_fold"])
    missing_hours = pd.to_numeric(operational["op_training_hours"],
                                  errors="coerce").isna()
    operational.loc[missing_hours, "op_training_hours"] = (
        pd.to_numeric(train_seconds[missing_hours], errors="coerce") / 3600.0)
    operational["op_output_tokens"] = first_column(operational,
        ["mean_completion_tokens", "median_completion_tokens",
         "mean_output_tokens", "output_tokens"])
    operational["op_median_seconds"] = first_column(operational, ["median_seconds"])
    operational["op_p95_seconds"] = first_column(operational, ["p95_seconds"])
    operational = operational.sort_values("arm").reset_index(drop=True)
    table10 = pd.DataFrame({"Arm": operational["arm"]})
    for source_column, header, kind in [
            ("op_total_parameters", "Total parameters", "count"),
            ("op_trainable_parameters", "Trainable parameters", "count"),
            ("op_peak_gpu_gb", "Peak GPU GB", "metric"),
            ("op_training_hours", "Training h/fold", "metric"),
            ("op_median_seconds", "Median s/image", "seconds"),
            ("op_p95_seconds", "p95 s/image", "seconds"),
            ("op_output_tokens", "Output tokens", "count")]:
        table10[header] = [sd.format_number(v, kind) for v in operational[source_column]]
    emit("table10_operational_cost", table10,
         "Operational cost per arm: parameter counts, peak GPU memory, wall-clock training "
         "time per fold, inference latency, and output token count. A dash means the owning "
         "notebook did not record that quantity for that arm; the gate reports those gaps "
         "without substituting a fabricated value.", "tab:operational")

# ---- Table S2: augmentation policy ---------------------------------------------------------
augmentation = pd.DataFrame([
    {"Setting": "Horizontal flip", "Value": "FORBIDDEN",
     "Reason": "mRALE is per-lung and laterality-specific; a flip relabels right as left"},
    {"Setting": "Vertical flip / 180 deg rotation", "Value": "FORBIDDEN",
     "Reason": "produces anatomically impossible radiographs"},
    {"Setting": "Elastic / grid distortion", "Value": "FORBIDDEN",
     "Reason": "alters the opacity extent the label measures"},
    {"Setting": "Random rotation", "Value": "+/- 7 degrees",
     "Reason": "positioning variation seen in portable studies"},
    {"Setting": "Random resized crop", "Value": "scale 0.90-1.00",
     "Reason": "small framing variation; a tighter crop would remove scored lung"},
    {"Setting": "Brightness / contrast jitter", "Value": "+/- 10%",
     "Reason": "acquisition variation, below the level that changes density judgement"},
    {"Setting": "CLAHE", "Value": "OFF in the primary pipeline",
     "Reason": "evaluated only as the E9c ablation arm"},
    {"Setting": "Primary intensity scaling",
     "Value": "per-image percentile clip, then min-max to [0,1]",
     "Reason": "fixed primary preprocessing in protocol Section 4.3"},
    {"Setting": "Test-time cohort re-normalisation",
     "Value": "OFF in the primary pipeline",
     "Reason": "mapping X2 to locked internal training statistics is an E9c label-free arm"},
])
emit("tableS2_augmentation_policy", augmentation,
     "Augmentation policy, including the forbidden list. The forbidden entries are the ones "
     "that matter: horizontal flip is standard practice in chest-radiograph pipelines and is "
     "invalid here because mRALE is scored per lung.", "tab:augmentation")

# ---- Direct protocol-map tables -----------------------------------------------------------
DIRECT_TABLE_SOURCES = {}
for name, frame, caption, label, source_name in [
        ("table1_cohort_composition", COHORT_TABLE,
         "Cohort and fold composition, copied without re-computation from NB 02/03.",
         "tab:cohort", COHORT_TABLE_SOURCE),
        ("table11_entity_probe_coefficients", PROBE_COEFFICIENTS,
         "BiomedCLIP entity-probe coefficients and across-fold stability from NB 08.",
         "tab:entity-probe", "probe_coefficients.csv (NB 08)"),
        ("tableS1_hyperparameter_provenance", HYPERPARAMETERS,
         "Hyperparameter provenance: sweep-selected values and inherited tested defaults.",
         "tab:hyperparameters", "sweep_selection.json files (NB 09-10)"),
        ("tableS3_model_registry", MODEL_REGISTRY,
         "Model registry with pinned revisions and parameter counts from NB 00.",
         "tab:model-registry", "model_registry.csv (NB 00)")]:
    if len(frame):
        direct = frame.reset_index(drop=True).copy()
        DIRECT_TABLE_SOURCES[name] = (source_name, direct)
        emit(name, direct, caption, label)

## 5. Figures 5 and 7, and the acronym table

Figure 3, 4 and 6 are produced by NB 18 and NB 20 and are copied here so every camera-ready
asset sits in one directory. Figures 5 and 7 are built here.

The acronym table (Appendix A) is extracted from the manuscript source when one is present, and
its first-use ordering is checked — R1.8 asked for that specifically.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 21 before code cell 11")

copied = []
nb18_figure_dir = NB18_DIR / "figures"
if nb18_figure_dir.is_dir():
    for path in sorted(nb18_figure_dir.glob("*")):
        # NB 18 plots must remain vector. PNG is reserved for radiographic case panels.
        if path.suffix.lower() in {".pdf", ".svg"}:
            target = FIGURE_DIR / path.name
            shutil.copy2(path, target)
            copied.append(target)

# Figure 6 may copy only panels named by NB 20's locked selection. Globbing the whole
# directory would let stale panels survive into the manuscript package.
panel_source_dir = NB20_DIR / "case_panels"
expected_case_panels, missing_case_panels, panel_manifest_rows = [], [], []
for row in CASE_SELECTION.to_dict("records"):
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", Path(str(row["filename"])).stem)
    key_hash = hashlib.sha256(str(row["image_key"]).encode("utf-8")).hexdigest()[:10]
    source = panel_source_dir / f"panel_{key_hash}_{stem}.png"
    expected_case_panels.append(source)
    panel_manifest_rows.append({
        "image_key": row["image_key"], "filename": row["filename"],
        "gt_band": row.get("gt_band"), "correct": row.get("correct"),
        "selection_reasons": row.get("selection_reasons"),
        "selection_hash": selection_hash, "source_panel": str(source),
        "exported": source.is_file()})
    if not source.is_file():
        missing_case_panels.append(str(row["image_key"]))
        continue
    target = FIGURE_DIR / source.name
    shutil.copy2(source, target)
    copied.append(target)

pd.DataFrame(panel_manifest_rows).to_csv(
    NB21_DIR / "figure6_case_panel_manifest.csv", index=False)
taxonomy_is_final = bool(kappa_result.get("available")
                         and kappa_result.get("taxonomy_final_available"))
captions["fig6"] = (
    "**Figure 6.** Qualitative case panels selected by the pre-registered E10 rule "
    f"(rule hash `{nb20_config.get('sampling_rule_hash')}`, selection hash "
    f"`{selection_hash}`). Each panel shows the radiograph, locked lung boxes, per-lung "
    "scores, agent outputs, PCR decision, and the reasoner's justification. Missing "
    "justifications remain visible as missing outputs rather than being removed.")
print(f"Copied {len(copied)} locked figure asset(s) from NB 18 and NB 20")

figure_paths = list(copied)
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 8.5, "axes.spines.top": False, "axes.spines.right": False,
                         "figure.dpi": 150, "savefig.bbox": "tight", "pdf.fonttype": 42,
                         "svg.fonttype": "none"})

    # ---- Figure 5: sensitivity tornado ----------------------------------------------------
    if len(SENSITIVITY) and "mae" in SENSITIVITY.columns:
        reference = float(SENSITIVITY["mae"].median())
        sensitivity_family = (SENSITIVITY["family"] if "family" in SENSITIVITY
                              else pd.Series("E6", index=SENSITIVITY.index))
        grouped = SENSITIVITY.groupby(sensitivity_family)["mae"].agg(["min", "max"])
        grouped["span"] = grouped["max"] - grouped["min"]
        grouped = grouped.sort_values("span")
        figure, axis = plt.subplots(figsize=(6.2, max(2.2, 0.42 * len(grouped) + 1.0)))
        positions = np.arange(len(grouped))
        axis.barh(positions, grouped["max"] - grouped["min"], left=grouped["min"], height=0.55)
        axis.axvline(reference, color="k", linestyle="--", linewidth=0.9,
                     label=f"median across arms ({sd.format_number(reference)})")
        axis.set_yticks(positions); axis.set_yticklabels(grouped.index)
        axis.set_xlabel("mRALE MAE across the arms in each sensitivity family")
        axis.set_title("Sensitivity ranges (exploratory)")
        axis.legend(fontsize=7, frameon=False)
        for suffix in ["pdf", "svg"]:
            path = FIGURE_DIR / f"fig5_sensitivity_tornado.{suffix}"
            figure.savefig(path); figure_paths.append(path)
        plt.close(figure)
        captions["fig5"] = (
            "**Figure 5.** Range of pooled mRALE MAE spanned by each sensitivity family "
            "(E5/E6). A family whose span is narrower than the confidence interval in Table 2 "
            "is evidence that the headline result does not depend on that choice. Exploratory: "
            "no multiplicity adjustment applies.")

    # ---- Figure 7: failure taxonomy ---------------------------------------------------------
    if len(TAXONOMY) and "n" in TAXONOMY.columns:
        taxonomy = TAXONOMY[TAXONOMY["n"] > 0].sort_values("n")
        if len(taxonomy):
            figure, axis = plt.subplots(figsize=(6.2, max(2.2, 0.4 * len(taxonomy) + 1.0)))
            positions = np.arange(len(taxonomy))
            colours = ["tab:orange" if code == "pcr_label_vs_imaging_mismatch" else "tab:blue"
                       for code in taxonomy["code"]]
            axis.barh(positions, taxonomy["n"], color=colours, height=0.6)
            axis.set_yticks(positions)
            axis.set_yticklabels([c.replace("_", " ") for c in taxonomy["code"]])
            axis.set_xlabel("Cases in the pre-registered sample")
            axis.set_title("Failure taxonomy" if taxonomy_is_final
                           else "Automatic failure-taxonomy first pass")
            for suffix in ["pdf", "svg"]:
                path = FIGURE_DIR / f"fig7_failure_taxonomy.{suffix}"
                figure.savefig(path); figure_paths.append(path)
            plt.close(figure)
            taxonomy_status = (
                f"two independent human coders with adjudication; Cohen's kappa "
                f"{kappa_result.get('kappa'):.3f}" if taxonomy_is_final else
                "automatic first pass; dual human coding is not yet complete")
            captions["fig7"] = (
                f"**Figure 7.** Failure taxonomy over the pre-registered case sample "
                f"({taxonomy_status}). The "
                "orange bar is PCR-positive cases with an annotated-clean radiograph, where "
                "the model's mRALE estimate agreed with the opacity annotation. These are "
                "not mRALE-opacity errors when scored zero, but the COVID decision is still "
                "evaluated against PCR. The bar quantifies label/opacity discordance only.")
    print(f"Total figure assets: {len(figure_paths)}")
except ImportError:
    print("matplotlib unavailable; tables are complete and figures can be rendered later.")

# ---- Appendix A: acronyms ------------------------------------------------------------------
ACRONYMS = [
    ("AUROC", "area under the receiver operating characteristic curve"),
    ("AUPRC", "area under the precision-recall curve"),
    ("CI", "confidence interval"), ("CLAHE", "contrast-limited adaptive histogram equalization"),
    ("CXR", "chest radiograph"), ("ECE", "expected calibration error"),
    ("IoU", "intersection over union"), ("LoRA", "low-rank adaptation"),
    ("MAE", "mean absolute error"), ("MCC", "Matthews correlation coefficient"),
    ("mRALE", "modified Radiographic Assessment of Lung Edema"),
    ("MIDRC", "Medical Imaging and Data Resource Center"),
    ("PCR", "polymerase chain reaction"), ("QWK", "quadratic-weighted kappa"),
    ("RALO", "Radiographic Assessment of Lung Oedema"),
    ("RMSE", "root mean squared error"), ("SSL", "self-supervised learning"),
    ("TOST", "two one-sided tests"), ("VLM", "vision-language model"),
]
acronyms = pd.DataFrame(ACRONYMS, columns=["Acronym", "Expansion"])

manuscript_sources = sorted((PROJECT_ROOT / "new_paper").glob("*.txt")) + \
    sorted((PROJECT_ROOT / "new_paper").glob("*.md")) \
    if (PROJECT_ROOT / "new_paper").is_dir() else []
first_use, ordering_problems = {}, []
if manuscript_sources:
    text = "\n".join(p.read_text(encoding="utf-8", errors="ignore")
                     for p in manuscript_sources)
    for acronym, expansion in ACRONYMS:
        position = text.find(acronym)
        expansion_position = text.lower().find(expansion.lower())
        first_use[acronym] = position
        if position >= 0 and expansion_position >= 0 and expansion_position > position:
            ordering_problems.append(
                f"{acronym} is used at character {position} before it is expanded at "
                f"{expansion_position}")
        elif position >= 0 and expansion_position < 0:
            ordering_problems.append(f"{acronym} is used but never expanded")
    acronyms["First use (char)"] = [first_use.get(a, -1) for a, _ in ACRONYMS]
    acronyms["Defined before use"] = [
        "" if any(problem.startswith(f"{a} ") for problem in ordering_problems)
        else "yes" for a, _ in ACRONYMS]
    print(f"\nChecked acronym first use against {len(manuscript_sources)} manuscript file(s)")
    if ordering_problems:
        print(f"  {len(ordering_problems)} ordering problem(s):")
        for problem in ordering_problems[:8]:
            print(f"    {problem}")
    else:
        print("  Every acronym is expanded before or at its first use (R1.8).")
else:
    print("\nNo manuscript source found under new_paper/; the acronym table is emitted without "
          "a first-use check.")
acronyms.to_csv(NB21_DIR / "acronym_table.csv", index=False)

## 6. Supplementary workbook and captions

One `.xlsx` with a sheet per table plus the two locked artifacts, so a reviewer can check any
figure in the paper against its source without a notebook kernel.

Captions are collected from the `emit` calls above rather than written separately — a caption
and its table cannot drift apart if the caption is what generated the table's file.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 21 before code cell 13")

workbook_path = NB21_DIR / "supplementary_results.xlsx"
try:
    with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
        workbook_sheets = []

        def write_sheet(frame, name):
            if frame is not None and len(frame):
                sheet = name[:31]
                if sheet in workbook_sheets:
                    raise RuntimeError(f"Duplicate Excel sheet name after truncation: {sheet}")
                frame.to_excel(writer, sheet_name=sheet, index=False)
                workbook_sheets.append(sheet)

        write_sheet(ALL_METRICS, "all_metrics_with_ci")
        write_sheet(PAIRED, "paired_comparisons_final")
        for name, frame in generated_tables.items():
            write_sheet(frame, name)
        for frame, name in [(SUBGROUPS, "subgroups_raw"), (EXTERNAL, "external_raw"),
                            (E9C, "threshold_transfer_raw"),
                            (CASE_SELECTION, "case_selection_raw"),
                            (TAXONOMY, "failure_taxonomy_raw"),
                            (GROUNDING, "grounding_raw")]:
            write_sheet(frame, name)
    print(f"supplementary_results.xlsx: {len(workbook_sheets)} sheets")
except ImportError:
    workbook_path = None
    print("openpyxl unavailable; the per-table CSVs in tables/ are the complete supplement.")

caption_lines = []
for name, caption in captions.items():
    title = name.replace("_", " ")
    caption_lines.append(f"**{title}.** {caption}" if not caption.startswith("**")
                         else caption)
(NB21_DIR / "captions.md").write_text("\n\n".join(caption_lines) + "\n", encoding="utf-8")
print(f"captions.md: {len(captions)} captions")

## 7. Traceability — the gate this notebook exists for

Every numeric cell is checked against the identity-matched row of its owning locked artifact.
A global value match is insufficient: the same rounded number in another arm or metric does not
establish provenance. A cell with no identity-matched source is a blocking failure.
supposed to prevent.

Derived quantities that are *not* metrics (row counts, policy text, acronym expansions) are
exempt by construction: they come from columns declared non-numeric below, rather than from a
list of pardoned exceptions that grows every time the gate complains.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 21 before code cell 15")

# Columns that carry text rather than measurements. Everything else is checked -- including
# counts, coverages and p-values, because those are measurements too.
NON_METRIC_COLUMNS = {"Arm", "Roster arm", "View / arm", "Aggregation method", "Comparison",
                      "Endpoint", "Family", "Test", "Paired unit", "Cohort",
                      "Sub-cohort", "Adapter", "Adapter provenance",
                      "Field", "Value", "Setting", "Reason", "Acronym", "Expansion",
                      "Excludes pooled", "Defined before use", "First use (char)"}

NUMBER = re.compile(r"-?\d+(?:\.\d+)?")


def matching_rows(frame, **keys):
    """Return the exact upstream row identified by the displayed row labels."""
    if not len(frame):
        return frame
    mask = pd.Series(True, index=frame.index)
    for column, value in keys.items():
        if column not in frame.columns:
            return frame.iloc[0:0]
        expected = "" if pd.isna(value) else str(value)
        mask &= frame[column].fillna("").astype(str) == expected
    return frame[mask]


def locked_rows_for(table_name, row_index):
    """Resolve a table row to its identity-matched locked upstream row(s)."""
    shown = generated_tables[table_name].iloc[row_index]
    if table_name in DIRECT_TABLE_SOURCES:
        source_name, source_frame = DIRECT_TABLE_SOURCES[table_name]
        return [(source_name, source_frame.iloc[[row_index]])]
    if table_name == "table2_main_comparison":
        return [("all_metrics_with_ci.csv (NB 17)", matching_rows(main, arm=shown["Arm"]))]
    if table_name == "table3_roster_ablation":
        arm = shown["Roster arm"]
        return [("all_metrics_with_ci.csv (NB 17)", matching_rows(roster, arm=arm)),
                ("paired_comparisons_final.csv (NB 19)",
                 matching_rows(PAIRED, arm_a=arm, arm_b=REFERENCE_ARM, family="F2",
                               endpoint="mRALE MAE",
                               reported_p_key="p_bootstrap"))]
    if table_name == "table4_localization":
        return [("all_metrics_with_ci.csv (NB 17)",
                 matching_rows(localization, arm=shown["View / arm"]))]
    if table_name == "table5_fusion_vs_reasoning":
        return [("all_metrics_with_ci.csv (NB 17)",
                 matching_rows(fusion, arm=shown["Aggregation method"]))]
    if table_name == "table6_sensitivity_exploratory":
        return [("e6_sensitivity_grid.csv (NB 16)",
                 matching_rows(sensitivity, arm=shown["Arm"], family=shown["Family"]))]
    if table_name == "table7_paired_tests":
        return [("paired_comparisons_final.csv (NB 19)", matching_rows(
            PAIRED, comparison=shown["Comparison"], endpoint=shown["Endpoint"],
            family=shown["Family"], test=shown["Test"],
            paired_unit=shown["Paired unit"]))]
    if table_name == "table8_external_validation":
        keys = {"cohort": shown["Cohort"], "arm": shown["Arm"]}
        if "subcohort" in external.columns:
            keys["subcohort"] = shown["Sub-cohort"]
        if "adapter" in external.columns:
            keys["adapter"] = shown["Adapter"]
        return [("external_metrics.csv (NB 19)", matching_rows(external, **keys))]
    if table_name == "table8b_x2_threshold_transfer":
        return [("e9c_threshold_transfer.csv (NB 19)", matching_rows(
            e9c_headline, cohort=shown["Cohort"], arm=shown["Arm"]))]
    if table_name == "table9_subgroups":
        return [("subgroup_metrics.csv (NB 19)", matching_rows(
            subgroups, field=shown["Field"], value=shown["Value"]))]
    if table_name == "table10_operational_cost":
        return [("identity-matched operational row (summary/config/journal)",
                 matching_rows(operational_locked, arm=shown["Arm"]))]
    return []


def traced(displayed, is_percent, locked_rows, token, is_upper_bound=False):
    """Match only within the identity-resolved upstream row, never the global value pool."""
    places = len(token.split(".")[1]) if "." in token else 0
    for source_name, frame in locked_rows:
        for column in frame.columns:
            raw_values = list(pd.to_numeric(frame[column], errors="coerce").dropna())
            if (source_name.startswith("sweep_selection.json") and column == "Value"):
                for raw_cell in frame[column]:
                    raw_values.extend(float(token) for token in NUMBER.findall(str(raw_cell)))
            for raw in raw_values:
                raw = float(raw)
                if not math.isfinite(raw):
                    continue
                if is_percent:
                    candidate = raw * 100.0
                    matches = round(candidate, places) == round(displayed, places)
                elif is_upper_bound:
                    matches = (displayed == 0.001 and 0 <= raw < displayed)
                else:
                    candidates = [raw]
                    # Some arm summaries store parameter counts in millions; Table 10
                    # displays absolute counts. This declared unit conversion is traceable.
                    if str(column).endswith("_M"):
                        candidates.append(raw * 1e6)
                    # Training summaries are stored in seconds; Table 10 declares hours.
                    if "train_seconds" in str(column):
                        candidates.append(raw / 3600.0)
                    matches = any(round(candidate, places) == round(displayed, places)
                                  for candidate in candidates)
                if matches:
                    return source_name
    return None


traceability_rows, untraceable = [], []
for name, frame in generated_tables.items():
    for column in frame.columns:
        if name in DIRECT_TABLE_SOURCES and not (
                name == "tableS1_hyperparameter_provenance" and column == "Value"):
            _, direct_source = DIRECT_TABLE_SOURCES[name]
            if not pd.to_numeric(direct_source[column], errors="coerce").notna().any():
                continue
        if column in NON_METRIC_COLUMNS and not (
                name == "tableS1_hyperparameter_provenance" and column == "Value"):
            continue
        for row_index, raw in enumerate(frame[column]):
            text = str(raw)
            if text.strip() in {"--", "", "unadjusted", "yes", "nan", "None"}:
                continue
            is_percent = "%" in text
            # Thousands separators must go before the regex runs: "1,092" would otherwise be
            # read as the two numbers 1 and 092, and the second traces to nothing.
            for match in NUMBER.findall(text.replace(",", "")):
                number = float(match)
                row_sources = locked_rows_for(name, row_index)
                source = traced(number, is_percent, row_sources, match,
                                is_upper_bound=("<" in text))
                traceability_rows.append({"table": name, "column": column, "row": row_index,
                                          "value": number, "percent": is_percent,
                                          "traced": source is not None,
                                          "source": source or ""})
                if source is None:
                    untraceable.append((name, column, row_index, number))

traceability = pd.DataFrame(traceability_rows)
traceability.to_csv(NB21_DIR / "traceability.csv", index=False)
if len(traceability):
    n_traced = int(traceability["traced"].sum())
    print(f"Traceability: {n_traced:,} of {len(traceability):,} numeric cells "
          f"({n_traced / len(traceability):.1%}) resolve to a locked upstream artifact.")
    print("  by source artifact:")
    for source, count in traceability[traceability["traced"]]["source"].value_counts().items():
        print(f"    {count:>6,}  {source}")
    if untraceable:
        print(f"  {len(untraceable)} untraceable value(s), e.g.:")
        for entry in untraceable[:8]:
            print(f"    {entry[0]} / {entry[1]} row {entry[2]}: {entry[3]}")
else:
    print("No numeric cells to trace.")

## 8. Run configuration and gate

Blocking conditions:

1. **NB 17--20 all passed, and their reference-arm and artifact fingerprints agree.**
2. **Every result number traces to its identity-matched locked source row.**
3. **The complete protocol table map exists, including the E9c transferred-threshold headline.**
4. **NB 19's final Holm families agree with the exported comparison table.**
5. **Every locked NB 20 case has its exact fingerprinted panel and a Figure 6 caption.**
6. **Table 1 covers all cohorts and Table 10 contains every required operational field.**

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 21 before code cell 17")

failures, warnings = [], []

if untraceable:
    examples = "; ".join(f"{t}/{c} row {r} = {v}" for t, c, r, v in untraceable[:5])
    failures.append(
        f"{len(untraceable)} numeric cell(s) do not trace to any locked upstream artifact: "
        f"{examples}. Either the value was typed here, or it was derived in this notebook "
        "rather than read from the notebook that computed it. Both mean the manuscript could "
        "carry a number nobody can reproduce.")
elif len(traceability):
    print(f"TRACEABILITY VERIFIED: all {len(traceability):,} numeric cells resolve to a locked "
          "upstream artifact.")

if "table2_main_comparison" not in generated_tables:
    failures.append("Table 2 (the main comparison) was not generated.")
elif REFERENCE_ARM not in set(generated_tables["table2_main_comparison"]["Arm"]):
    failures.append(f"Table 2 does not contain the reference arm {REFERENCE_ARM}.")
else:
    print(f"Table 2 contains the reference arm {REFERENCE_ARM}.")

# The cross-site PCR headline is the transferred internal operating point, not a threshold
# selected on X2. If NB 19 produced it, NB 21 must export it as a manuscript table.
x2_present = (len(EXTERNAL) and "cohort" in EXTERNAL
              and EXTERNAL["cohort"].astype(str).str.upper().eq("X2").any())
if x2_present and not len(e9c_headline):
    failures.append("NB 19 contains X2 external results but no E9c HEADLINE row at the "
                    "internally transferred operating point.")
if len(e9c_headline) and "table8b_x2_threshold_transfer" not in generated_tables:
    failures.append("Table 8b was not generated from NB 19's E9c HEADLINE rows.")
elif len(e9c_headline) and (
        len(generated_tables["table8b_x2_threshold_transfer"]) != len(e9c_headline)):
    failures.append("Table 8b row count disagrees with NB 19's E9c HEADLINE rows.")

# Figure 6 is a deterministic export of NB 20's locked qualitative selection.
panel_manifest_path = NB21_DIR / "figure6_case_panel_manifest.csv"
panel_manifest = (pd.read_csv(panel_manifest_path) if panel_manifest_path.is_file()
                  else pd.DataFrame())
if missing_case_panels:
    failures.append(f"{len(missing_case_panels)}/{len(CASE_SELECTION)} locked NB 20 case "
                    "panel(s) are missing; Figure 6 would silently omit selected cases.")
if len(panel_manifest) != len(CASE_SELECTION):
    failures.append("Figure 6 panel manifest does not cover every locked NB 20 case.")
elif len(panel_manifest):
    if set(panel_manifest["selection_hash"].astype(str)) != {selection_hash}:
        failures.append("Figure 6 panel manifest has a stale NB 20 selection hash.")
    exported = panel_manifest["exported"].astype(str).str.lower().eq("true")
    if not exported.all():
        failures.append("Figure 6 panel manifest contains unexported selected cases.")
if "fig6" not in captions:
    failures.append("Figure 6 caption is missing.")

# ---- Completeness and presentation checks ---------------------------------------------------
expected = {"table2_main_comparison": "Table 2", "table3_roster_ablation": "Table 3",
            "table4_localization": "Table 4", "table5_fusion_vs_reasoning": "Table 5",
            "table6_sensitivity_exploratory": "Table 6", "table7_paired_tests": "Table 7",
            "table8_external_validation": "Table 8", "table9_subgroups": "Table 9",
            "table10_operational_cost": "Table 10",
            "table11_entity_probe_coefficients": "Table 11",
            "table1_cohort_composition": "Table 1",
            "tableS1_hyperparameter_provenance": "Table S1",
            "tableS2_augmentation_policy": "Table S2",
            "tableS3_model_registry": "Table S3"}
if len(e9c_headline):
    expected["table8b_x2_threshold_transfer"] = "Table 8b"
missing_tables = [label for name, label in expected.items() if name not in generated_tables]
if missing_tables:
    failures.append(f"Required protocol-map artifacts were not generated: {missing_tables}. "
                    "Run the owning upstream notebooks before camera-ready export.")
if "internal only" in COHORT_TABLE_SOURCE:
    failures.append("Table 1 contains only the internal cohort. NB 03 must emit "
                    "cohort_composition_table1_full.csv for the all-cohort table.")

if "table10_operational_cost" in generated_tables:
    cost = generated_tables["table10_operational_cost"]
    required_cost = ["Total parameters", "Trainable parameters", "Peak GPU GB",
                     "Training h/fold", "Median s/image", "p95 s/image",
                     "Output tokens"]
    wholly_missing = [column for column in required_cost
                     if column not in cost or cost[column].astype(str).isin(["--", "nan", ""]).all()]
    if wholly_missing:
        failures.append(f"Table 10 lacks required operational measurements: {wholly_missing}.")
    partial = {column: int(cost[column].astype(str).isin(["--", "nan", ""]).sum())
               for column in required_cost if column in cost}
    partial = {column: count for column, count in partial.items() if count}
    if partial:
        warnings.append(f"Table 10 has partially missing per-arm measurements: {partial}. "
                        "Use N/A only when the quantity is structurally inapplicable.")
if operational_ambiguities:
    examples = {arm: sources for arm, sources in list(operational_ambiguities.items())[:5]}
    failures.append("Table 10 found duplicate rows for an arm across the discovered "
                    "arm_summary.csv artifacts; "
                    f"source selection would be ambiguous: {examples}.")

if ordering_problems:
    warnings.append(f"{len(ordering_problems)} acronym(s) used before expansion or never "
                    f"expanded (R1.8): {ordering_problems[:4]}")
if not manuscript_sources:
    warnings.append("No manuscript source was found, so the acronym first-use check (R1.8) did "
                    "not run.")

vector = [p for p in figure_paths if p.suffix.lower() in {".pdf", ".svg"}]
raster = [p for p in figure_paths if p.suffix.lower() == ".png"]
if not vector:
    warnings.append("No vector figures were produced. R1.9 complained the figures were "
                    "illegible; PDF/SVG is the fix.")
if raster:
    warnings.append(f"{len(raster)} raster asset(s) (case panels) are included. Panels are "
                    "photographs and are correctly raster, but no plotted figure should be.")

# Validate the final multiplicity artifact, not just the presence of an adjusted column.
final_family_counts = multiplicity_final.get("family_counts", {})
if len(PAIRED):
    required_p_columns = {"family", "p_raw", "p_adjusted", "p_reportable"}
    missing_p_columns = sorted(required_p_columns - set(PAIRED.columns))
    if missing_p_columns:
        failures.append(f"Final paired comparison lacks columns {missing_p_columns}.")
    else:
        finite_p = PAIRED[pd.to_numeric(PAIRED["p_raw"], errors="coerce").notna()].copy()
        observed_counts = finite_p.groupby("family").size().astype(int).to_dict()
        for family in sorted(set(observed_counts) | set(final_family_counts)):
            if int(observed_counts.get(family, 0)) != int(final_family_counts.get(family, 0)):
                failures.append(
                    f"Multiplicity family {family} has {observed_counts.get(family, 0)} "
                    f"finite tests in paired_comparisons_final.csv but "
                    f"multiplicity_families_final.json declares "
                    f"{final_family_counts.get(family, 0)}.")
        confirmatory = finite_p[finite_p["family"].astype(str) != "EXPLORATORY"]
        adjusted = pd.to_numeric(confirmatory["p_adjusted"], errors="coerce")
        reportable = pd.to_numeric(confirmatory["p_reportable"], errors="coerce")
        if adjusted.isna().any() or reportable.isna().any():
            failures.append("Final paired-comparison artifact has confirmatory rows without "
                            "Holm-adjusted/reportable p-values.")
        elif not np.allclose(adjusted.to_numpy(), reportable.to_numpy(),
                             rtol=0, atol=1e-12):
            failures.append("Confirmatory p_reportable values are not the final Holm-adjusted "
                            "p-values.")
        exploratory = finite_p[finite_p["family"].astype(str) == "EXPLORATORY"]
        if len(exploratory):
            raw = pd.to_numeric(exploratory["p_raw"], errors="coerce")
            shown = pd.to_numeric(exploratory["p_reportable"], errors="coerce")
            if shown.isna().any() or not np.allclose(raw.to_numpy(), shown.to_numpy(),
                                                     rtol=0, atol=1e-12):
                failures.append("Exploratory p_reportable values must equal their unadjusted "
                                "p_raw values.")

taxonomy_status = ("dual_coded_final" if taxonomy_is_final
                   else "automatic_first_pass")
taxonomy_total = int(pd.to_numeric(TAXONOMY["n"], errors="coerce").fillna(0).sum())
if taxonomy_total != len(CASE_SELECTION):
    failures.append(f"Failure-taxonomy counts cover {taxonomy_total}/{len(CASE_SELECTION)} "
                    "locked cases.")
if taxonomy_is_final:
    if "coder" not in TAXONOMY or not TAXONOMY["coder"].astype(str).str.contains(
            "two independent humans", case=False, na=False).all():
        failures.append("NB 20 claims a final dual-coded taxonomy but its taxonomy rows do "
                        "not carry the two-human provenance.")
else:
    warnings.append("Figure 7 and its workbook table are an automatic failure-taxonomy "
                    "first pass; do not describe them as dual-coded results.")

missing_captions = [name for name in generated_tables if name not in captions]
if missing_captions:
    failures.append(f"Generated tables without captions: {missing_captions}.")

sd.write_json_atomic(NB21_DIR / "run_config.json", sd.provenance_stamp(
    "21_figures_tables_and_exports.ipynb",
    {"reference_arm": REFERENCE_ARM,
     "upstream_gates": {name: {"path": entry["path"], "passed": True}
                        for name, entry in UPSTREAM_GATES.items()},
     "bootstrap_fingerprint": expected_bootstrap,
     "nb20_selection_hash": selection_hash,
     "nb20_case_table_fingerprint": case_fingerprint,
     "taxonomy_status": taxonomy_status,
     "final_family_counts": final_family_counts,
     "e9c_headline_rows": int(len(e9c_headline)),
     "case_panels_expected": int(len(CASE_SELECTION)),
     "case_panels_missing": missing_case_panels,
     "tables_generated": list(generated_tables),
     "n_figure_assets": len(figure_paths),
     "vector_figures": [p.name for p in vector],
     "workbook": (str(workbook_path) if workbook_path else None),
     "n_numeric_cells": int(len(traceability)),
     "n_untraceable": len(untraceable),
     "acronym_ordering_problems": ordering_problems,
     "formatting": ("every float passes through stage_d_stats.format_number; best in bold and "
                    "second best underlined, chosen by each metric's direction"),
     "traceability_sources": list(LOCKED_SOURCES),
     "traceability_rule": ("every numeric cell must resolve within the identity-matched "
                           "upstream row for that table row; global value collisions are not "
                           "accepted; traceability.csv records the source per cell")}))


def report(title, messages):
    print(title)
    for message in messages or []:
        print("  -", message)
    if not messages:
        print("  none")


print()
report("WARNINGS", warnings)
print()
report("FAILURES", failures)
sd.write_json_atomic(NB21_DIR / "gate_nb21.json",
                     {"passed": not failures, "failures": failures,
                      "warnings": warnings, "reference_arm": REFERENCE_ARM,
                      "bootstrap_fingerprint": expected_bootstrap,
                      "nb20_selection_hash": selection_hash,
                      "nb20_case_table_fingerprint": case_fingerprint,
                      "taxonomy_status": taxonomy_status,
                      "final_family_counts": final_family_counts,
                      "e9c_headline_rows": int(len(e9c_headline)),
                      "case_panels_expected": int(len(CASE_SELECTION)),
                      "case_panels_missing": missing_case_panels,
                      "upstream_gates": {name: entry["path"]
                                           for name, entry in UPSTREAM_GATES.items()}})
if failures:
    detail = "\n".join(f"  [{i + 1}] {m}" for i, m in enumerate(failures))
    raise AssertionError(f"NB 21 gate failed with {len(failures)} blocking issue(s):\n{detail}")
print()
print("NB 21 gate: PASSED")
print()
print(f"{len(generated_tables)} tables and {len(figure_paths)} figure assets in {NB21_DIR}")